In [1]:
# --- Cell 1: Imports & helpers
from tqdm.notebook import tqdm  # use tqdm.auto if you run in a script 
import json, time, re, os, textwrap, uuid
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
import requests
import pandas as pd

OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "llama3.1")  # change if needed

def _clean(s: str) -> str:
    return (s or "").strip()

def _to_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return [x]


In [2]:
# --- Cell 2: Load a single JSON object or a JSONL file
def load_data(path: str) -> List[Dict[str, Any]]:
    """
    Accepts either:
      - a single JSON object file, or
      - a JSONL with one object per line.
    Returns a list of dicts.
    """
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(path)
    if p.suffix.lower() == ".jsonl":
        rows = []
        with p.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                rows.append(json.loads(line))
        return rows
    else:
        obj = json.loads(p.read_text(encoding="utf-8"))
        if isinstance(obj, list):
            return obj
        return [obj]

# Example:
# data = load_data("your_file.json")


In [17]:
# --- Cell 3: LLM judge via Ollama (chat endpoint)
JUDGE_SYSTEM_PROMPT = """\
You are a strict evaluator of semantic equivalence between a TARGET answer and a CANDIDATE answer.
Assess whether the candidate conveys the same meaning as the target in the given CONTEXT.
Focus on meaning, not wording. Be robust to synonyms and minor phrasing differences.

Return ONLY a valid compact JSON object with keys:
{"equivalent": true|false, "score": float in [0,1], "rationale": "short reason"}

- "equivalent" must be true if (and only if) the candidate would be accepted as correct.
- "score" is your confidence that the meanings match (1.0 exact; 0.0 not equivalent).
- "rationale" should be brief (<= 25 words).
"""

JUDGE_USER_TEMPLATE = """\
TARGET:
{target}

CANDIDATE:
{candidate}

Decide if CANDIDATE expresses the same meaning as TARGET.
"""


def ollama_chat_judge(
    model: str,
    context: str,
    target: str,
    candidate: str,
    host: str = OLLAMA_HOST,
    temperature: float = 0.0,
    max_retries: int = 3,
    retry_sleep: float = 1.0,
) -> Dict[str, Any]:
    """
    Calls Ollama /api/chat with a JSON-only response instruction.
    Returns dict with keys: equivalent (bool), score (float), rationale (str).
    """
    url = f"{host}/api/chat"
    messages = [
        {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
        {"role": "user", "content": JUDGE_USER_TEMPLATE.format(
            context=context,
            target=target,
            candidate=candidate
        )},
    ]

    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.post(
                url,
                json={
                    "model": model,
                    "messages": messages,
                    "stream": False,
                    "options": {
                        "temperature": temperature,
                    },
                },
                timeout=120
            )
            resp.raise_for_status()
            data = resp.json()
            content = data.get("message", {}).get("content", "").strip()

            # Try to extract JSON
            # Some models might wrap JSON in code fences – handle that
            json_str = content
            fence = re.search(r"\{.*\}", content, re.DOTALL)
            if fence:
                json_str = fence.group(0)

            parsed = json.loads(json_str)
            # validate
            eq = bool(parsed.get("equivalent"))
            score = float(parsed.get("score", 1.0 if eq else 0.0))
            rationale = _clean(parsed.get("rationale", ""))
            return {"equivalent": eq, "score": score, "rationale": rationale, "raw": content}
        except Exception as e:
            if attempt == max_retries:
                # final fallback
                return {"equivalent": False, "score": 0.0, "rationale": f"Parse/LLM error: {e}", "raw": ""}
            time.sleep(retry_sleep)


In [29]:
# --- Cell 4: Pairing and judging
def to_context(obj: Dict[str, Any]) -> str:
    # Prefer 'input' as-is; if it contains "Context:" and "Question:" it's already formatted.
    return _clean(obj.get("input", ""))

def extract_targets(obj: Dict[str, Any]) -> List[str]:
    # Your JSON shows "target" is a list of strings
    return [_clean(t) for t in _to_list(obj.get("target", [])) if _clean(t)]

def extract_candidates(obj: Dict[str, Any]) -> List[str]:
    # Your JSON shows "clarified_all_ans" is List[List[str]]
    cands = []
    blocks = _to_list(obj.get("clarified_all_ans", []))
    for block in blocks:
        for s in _to_list(block):
            s_clean = _clean(s)
            if s_clean:
                cands.append(s_clean)
    return cands

def best_equivalence_against_targets(
    context: str,
    targets: List[str],
    candidate: str,
    model: str = OLLAMA_MODEL,
    host: str = OLLAMA_HOST,
) -> Dict[str, Any]:
    """
    Ask the judge for candidate vs each target; keep the best (max score).
    """
    results = []
    for tgt in targets:
        r = ollama_chat_judge(model=model, context=context, target=tgt, candidate=candidate, host=host)
        r["target"] = tgt
        results.append(r)
    # choose best by score, break ties preferring equivalent=True
    results.sort(key=lambda d: (d["score"], d["equivalent"]), reverse=True)
    best = results[0]
    return {
        "best_equivalent": best["equivalent"],
        "best_score": best["score"],
        "best_rationale": best["rationale"],
        "best_target": best["target"],
        "judge_raw": best.get("raw", ""),
    }


In [30]:
# --- Cell 5: Batch runner
def evaluate_file(
    path: str,
    out_csv: Optional[str] = None,
    model: str = OLLAMA_MODEL,
    host: str = OLLAMA_HOST,
) -> pd.DataFrame:
    objs = load_data(path)
    out_rows: List[Dict[str, Any]] = []

    # total number of candidates for accurate progress tracking
    total_candidates = sum(
        len(extract_candidates(obj)) or 1 for obj in objs
    )

    with tqdm(total=total_candidates, desc="Judging answers", unit="cand") as pbar:
        for i, obj in enumerate(objs):
            uid = obj.get("id", f"ex-{i}")
            context = to_context(obj)
            targets = extract_targets(obj)
            candidates = extract_candidates(obj)

            if not targets or not candidates:
                out_rows.append({
                    "id": uid,
                    "candidate": "",
                    "best_equivalent": False,
                    "best_score": 0.0,
                    "best_rationale": "Missing targets or candidates",
                    "best_target": "",
                    "context": context,
                })
                pbar.update(1)
                continue

            for cand in candidates:
                best = best_equivalence_against_targets(context, targets, cand, model=model, host=host)
                out_rows.append({
                    "id": uid,
                    "candidate": cand,
                    "best_equivalent": best["best_equivalent"],
                    "best_score": best["best_score"],
                    "best_rationale": best["best_rationale"],
                    "best_target": best["best_target"],
                    "context": context,
                })
                pbar.update(1)

    df = pd.DataFrame(out_rows)
    if out_csv:
        Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(out_csv, index=False, encoding="utf-8")
    return df
# Example:
# df = evaluate_file("your_file.json", out_csv="judged_answers.csv")
# df.head()


In [31]:
# Replace/augment your ollama_chat_judge with a debug flag and shorter timeout
import time, re, json, datetime, requests

def ollama_chat_judge_debug(model, context, target, candidate, host="http://localhost:11434",
                            temperature=0.0, max_retries=2, retry_sleep=1.0, timeout_s=30, debug=True):
    url = host.rstrip("/") + "/api/chat"
    messages = [
        {"role": "system", "content": "Return only JSON: {\"equivalent\":true|false,\"score\":0..1,\"rationale\":\"...\"}"},
        {"role": "user", "content": f"CONTEXT:\n{context}\n\nTARGET:\n{target}\n\nCANDIDATE:\n{candidate}\n"}
    ]
    for attempt in range(1, max_retries+1):
        try:
            if debug:
                ts = datetime.datetime.now().strftime("%H:%M:%S")
                print(f"[{ts}] -> Ollama call {attempt}/{max_retries} | len(context)={len(context)}, len(cand)={len(candidate)}", flush=True)
            r = requests.post(url, json={"model":model,"messages":messages,"stream":False,"options":{"temperature":temperature}},
                              timeout=timeout_s)
            r.raise_for_status()
            content = r.json().get("message",{}).get("content","").strip()
            if debug:
                print(f"   <- {content[:120]}{'...' if len(content)>120 else ''}", flush=True)
            m = re.search(r"\{.*\}", content, re.DOTALL)
            parsed = json.loads(m.group(0) if m else content)
            return parsed
        except Exception as e:
            if attempt == max_retries:
                raise
            time.sleep(retry_sleep)

# Tiny smoke test (should print quickly)
_ = ollama_chat_judge_debug("llama3.1", "short ctx", "full union", "full communion")
print("Smoke test OK")


[22:06:38] -> Ollama call 1/2 | len(context)=9, len(cand)=14
   <- Here is the JSON output:

```
{
  "equivalent": false,
  "score": 0.5,
  "rationale": "The target 'full union' is not eq...
Smoke test OK


In [32]:
data = load_data("results/squadv2.json")

# View how many objects total
print(f"Total examples: {len(data)}")

# Take only the first 3 for testing
small_data = data[:1]

# Save this mini-sample to a temp file
import json
with open("sample.json", "w", encoding="utf-8") as f:
    json.dump(small_data, f, ensure_ascii=False, indent=2)

# Run evaluation on that smaller file
df_small = evaluate_file("sample.json", out_csv="sample_results.csv")
df_small.head()


Total examples: 40


Judging answers: 100%|█████████████████████████████████████████████| 50/50 [02:51<00:00,  3.43s/cand]


,id,candidate,best_equivalent,best_score,best_rationale,best_target,context
0,ex-0,The term that describes the areas where the fi...,False,0.8,The candidate describes a specific concept (ec...,full union,Context: The Roman Catholic Church canon law a...
1,ex-0,The correct answer is:\n\n**Particular Churche...,True,0.9,CANDIDATE accurately describes the concept of ...,full union,Context: The Roman Catholic Church canon law a...
2,ex-0,The answer is Ecclesiastical,False,0.0,"Candidate refers to a specific type of union, ...",full union,Context: The Roman Catholic Church canon law a...
3,ex-0,"The correct answer is ""Territories"". The quest...",False,0.8,The candidate describes a specific context (fi...,full union,Context: The Roman Catholic Church canon law a...
4,ex-0,"The correct answer is ""Rite"" or more specifica...",False,0.8,CANDIDATE is close but not exact; it mentions ...,full union,Context: The Roman Catholic Church canon law a...


In [33]:
df_small.iloc[4]['candidate']

'The correct answer is "Rite" or more specifically, "Eastern Rite" or simply "Oriental Rite" since there are Eastern (Greek) and Oriental (Syriac, Chaldean etc.) Churches. \n\nHowever, given the context of geographical overlap, it\'s likely that the term you\'re looking for is "Rite".\n\nThe five rites (groups) of churches in full union with the Roman Catholic Church are:\n\n1. Latin Rite\n2. Eastern Rite (Greek)\n3. Oriental Rite\n4. Maronite Rite\n5. Malankara Rite'

In [9]:
df = evaluate_file("results/squadv2.json", out_csv="judged_answers.csv")


Judging answers:   0%|                                                    | 0/1960 [00:00<?, ?cand/s]

[heartbeat] processed 0 candidates…


Judging answers:   1%|▌                                        | 25/1960 [01:18<1:38:15,  3.05s/cand]

[heartbeat] processed 25 candidates…


Judging answers:   3%|█                                        | 50/1960 [02:37<1:38:33,  3.10s/cand]

...

[heartbeat] processed 150 candidates…


Judging answers:   9%|███▌                                    | 175/1960 [08:34<1:21:52,  2.75s/cand]

[heartbeat] processed 175 candidates…


Judging answers:  10%|███▉                                    | 194/1960 [09:30<1:26:30,  2.94s/cand]


KeyboardInterrupt: 